# ML: Customer Segmentation (RFM + K-means)

Segments customers with source-anchored RFM features and deterministic Spark
K-means.

## Model contract
- Recency and `segmented_at` are anchored to the source as-of time, the latest
  Silver receipt timestamp shared by every required ML producer;
  `generated_at` records when the Gold output is published.
- RFM counts every purchase of a known customer up to the as-of time: store
  `SALE` receipts and online orders. Returns and anonymous purchases (null
  `customer_id`) are excluded, and customers with no purchase get no segment.
- Candidate selection and final fitting use a fixed seed and stable customer
  ordering.
- Raw K-means labels are remapped to canonical IDs by lexicographically sorting
  the fitted centroids in RFM feature order.
- The Gold output contains exactly one row per non-null customer.


In [ ]:
import os

import mlflow
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.sql import functions as F


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env(
    "MLFLOW_EXPERIMENT", default="customer_segmentation"
)

MIN_K = int(get_env("MIN_K", default="4"))
MAX_K = int(get_env("MAX_K", default="10"))
RANDOM_SEED = int(get_env("RANDOM_SEED", default="42"))

if MIN_K < 2 or MAX_K < MIN_K:
    raise ValueError("K-means bounds must satisfy 2 <= MIN_K <= MAX_K.")

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"K-means range: {MIN_K}-{MAX_K}; seed={RANDOM_SEED}")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_receipts', 'fact_online_order_headers')
ML_OUTPUT_CONTRACTS = {'customer_segments': [('customer_id', 'long', False),
                       ('cluster_id', 'int', False),
                       ('segment_label', 'string', False),
                       ('recency_days', 'double', False),
                       ('frequency', 'double', False),
                       ('monetary_value', 'double', False),
                       ('avg_order_value', 'double', False),
                       ('first_purchase_date', 'date', False),
                       ('last_purchase_date', 'date', False),
                       ('segmented_at', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)]}


In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time as _time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                _time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = _time.sleep,
    monotonic: Callable[[], float] = _time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# One writer per ML output: this notebook runs in more than one pipeline, and
# notebook 05 maintains the same table (see job_locks).
LOCK_WAIT_SECONDS = int(get_env("LOCK_WAIT_SECONDS", default="1800"))
JOB_LOCKS = JobLocks(
    DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}"),
    "08-ml-customer-segmentation",
    [ml_output_lock("customer_segments")],
)
if not JOB_LOCKS.acquire(wait_seconds=LOCK_WAIT_SECONDS):
    raise RuntimeError(f"ML output locked by another job: {JOB_LOCKS.blockers()}")

In [ ]:
# =============================================================================
# HELPERS
# =============================================================================


def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")


def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")


def save_gold(frame, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    frame.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(full_name)
    print(f"  {full_name}: {frame.count()} rows")


def canonical_cluster_mapping(centers):
    """Map arbitrary K-means labels to centroid-sorted canonical IDs."""
    indexed_centers = [
        (raw_id, tuple(float(value) for value in center))
        for raw_id, center in enumerate(centers)
    ]
    ordered = sorted(indexed_centers, key=lambda item: (item[1], item[0]))
    return {
        raw_id: canonical_id
        for canonical_id, (raw_id, _) in enumerate(ordered)
    }


def select_best_k(scores):
    """Select highest silhouette, breaking ties toward the smaller K."""
    if not scores:
        raise ValueError("At least one candidate K score is required.")
    return min(scores, key=lambda item: (-float(item[1]), int(item[0])))


def assert_unique_keys(frame, key_columns, context):
    duplicates = (
        frame.groupBy(*key_columns)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    )
    if duplicates:
        raise RuntimeError(f"{context} is not unique by {key_columns}.")


def purchase_amount():
    return F.coalesce(
        F.col("total_amount").cast("double"),
        F.col("total_cents").cast("double") / 100.0,
    )


def customer_purchases(receipts, online_orders, as_of_timestamp):
    """Store SALE receipts and online orders of known customers up to the as-of."""
    columns = [
        F.col("customer_id").cast("long").alias("customer_id"),
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        purchase_amount().alias("amount"),
    ]
    store = receipts.filter(F.col("receipt_type") == "SALE").select(*columns)
    online = online_orders.select(*columns)
    return store.unionByName(online).filter(
        F.col("customer_id").isNotNull()
        & (F.col("event_ts") <= F.lit(as_of_timestamp))
    )


def build_rfm(purchases, analysis_date):
    """Exactly one RFM row per customer with at least one purchase."""
    return (
        purchases.groupBy("customer_id")
        .agg(
            F.datediff(
                F.lit(analysis_date),
                F.max(F.to_date("event_ts")),
            ).alias("recency_days"),
            F.count(F.lit(1)).alias("frequency"),
            F.sum("amount").alias("monetary_value"),
            F.min(F.to_date("event_ts")).alias("first_purchase_date"),
            F.max(F.to_date("event_ts")).alias("last_purchase_date"),
        )
        .filter(
            F.col("recency_days").isNotNull()
            & (F.col("frequency") > 0)
            & (F.col("monetary_value") > 0)
        )
        .withColumn(
            "avg_order_value",
            F.col("monetary_value") / F.col("frequency"),
        )
    )


ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



---
## Step 1: Calculate RFM Metrics

Calculate RFM (Recency, Frequency, Monetary) metrics for each customer:
- **Recency**: Days since last purchase
- **Frequency**: Total number of transactions
- **Monetary**: Total spending amount

In [ ]:
print("=" * 60)
print("CALCULATING SOURCE-ANCHORED RFM METRICS")
print("=" * 60)

df_receipts = read_silver("fact_receipts")
df_online_orders = read_silver("fact_online_order_headers")
source_range = df_receipts.agg(
    F.min(F.col("event_ts").cast("timestamp")).alias("first_source_ts"),
    F.max(F.col("event_ts").cast("timestamp")).alias("analysis_timestamp"),
).first()
analysis_timestamp = source_range["analysis_timestamp"]
if analysis_timestamp is None:
    raise RuntimeError("fact_receipts contains no source timestamps.")
analysis_date = analysis_timestamp.date()
df_purchases = customer_purchases(
    df_receipts, df_online_orders, analysis_timestamp
)

print()
print(f"Analysis source timestamp: {analysis_timestamp}")
print(f"Customer purchases (store and online): {df_purchases.count():,}")
print(
    f"Source range: {source_range['first_source_ts']} to "
    f"{analysis_timestamp}"
)


In [ ]:
# Calculate exactly one RFM row per customer with a purchase.
df_rfm = build_rfm(df_purchases, analysis_date).orderBy("customer_id").cache()

assert_unique_keys(df_rfm, ["customer_id"], "RFM feature frame")
print()
print(f"Customers with RFM metrics: {df_rfm.count():,}")

print()
print("Deterministic sample RFM data:")
df_rfm.orderBy(
    F.rand(seed=RANDOM_SEED),
    F.col("customer_id"),
).limit(5).show()

print()
print("RFM Statistics:")
df_rfm.select(
    F.mean("recency_days").alias("avg_recency"),
    F.mean("frequency").alias("avg_frequency"),
    F.mean("monetary_value").alias("avg_monetary"),
    F.mean("avg_order_value").alias("avg_order_value"),
).show()


---
## Step 2: Feature Engineering & Scaling

Standardize RFM features for K-means clustering.

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING & SCALING")
print("=" * 60)

df_features = df_rfm.select(
    "customer_id",
    F.col("recency_days").cast("double").alias("recency_days"),
    F.col("frequency").cast("double").alias("frequency"),
    F.col("monetary_value").cast("double").alias("monetary_value"),
    F.col("avg_order_value").cast("double").alias("avg_order_value"),
    "first_purchase_date",
    "last_purchase_date",
).orderBy("customer_id")

assembler = VectorAssembler(
    inputCols=["recency_days", "frequency", "monetary_value"],
    outputCol="features_raw",
)
df_assembled = assembler.transform(df_features)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)
scaler_model = scaler.fit(df_assembled)
df_scaled = (
    scaler_model.transform(df_assembled)
    .orderBy("customer_id")
    .cache()
)
_ = df_scaled.count()

print()
print("Feature scaling complete.")
print(f"Scaled features ready: {df_scaled.count():,} customers")


---
## Step 3: Evaluate Candidate K Values

Evaluate candidate cluster counts using WCSS and silhouette, then select K automatically by highest silhouette.

In [ ]:
print("=" * 60)
print("EVALUATING CANDIDATE K VALUES")
print("=" * 60)

wcss_scores = []
silhouette_scores = []
evaluator = ClusteringEvaluator(
    predictionCol="candidate_cluster_id",
    featuresCol="features",
    metricName="silhouette",
)

customer_count = df_scaled.count()
max_candidate_k = min(MAX_K, customer_count - 1)
if max_candidate_k < MIN_K:
    raise RuntimeError(
        f"Need more than {MIN_K} customers for deterministic K-means search."
    )

for k_value in range(MIN_K, max_candidate_k + 1):
    candidate_model = KMeans(
        featuresCol="features",
        predictionCol="candidate_cluster_id",
        k=k_value,
        seed=RANDOM_SEED,
        maxIter=20,
    ).fit(df_scaled.orderBy("customer_id"))
    candidate_predictions = candidate_model.transform(df_scaled)
    wcss = float(candidate_model.summary.trainingCost)
    silhouette = float(evaluator.evaluate(candidate_predictions))
    wcss_scores.append((k_value, wcss))
    silhouette_scores.append((k_value, silhouette))
    print(
        f"K={k_value}: WCSS={wcss:,.2f}, "
        f"Silhouette={silhouette:.4f}"
    )

best_k, best_silhouette = select_best_k(silhouette_scores)
print(
    f"Selected K={best_k} by highest silhouette with lower-K tie break "
    f"({best_silhouette:.4f})."
)


---
## Step 4: Train K-means with Selected K

Train K-means using the K selected from the candidate search.

In [ ]:
print("=" * 60)
print("TRAINING DETERMINISTIC K-MEANS MODEL")
print("=" * 60)

OPTIMAL_K, selected_silhouette = select_best_k(silhouette_scores)
kmeans = KMeans(
    featuresCol="features",
    predictionCol="raw_cluster_id",
    k=OPTIMAL_K,
    seed=RANDOM_SEED,
    maxIter=50,
)
model = kmeans.fit(df_scaled.orderBy("customer_id"))
df_raw_clustered = model.transform(df_scaled)

canonical_mapping = canonical_cluster_mapping(model.clusterCenters())
df_cluster_mapping = spark.createDataFrame(
    sorted(canonical_mapping.items()),
    ["raw_cluster_id", "cluster_id"],
)
df_clustered = (
    df_raw_clustered.join(
        df_cluster_mapping,
        on="raw_cluster_id",
        how="inner",
    )
    .drop("raw_cluster_id")
    .orderBy("customer_id")
    .cache()
)
assert_unique_keys(df_clustered, ["customer_id"], "Cluster assignment frame")

print()
print("Model training complete.")
print(f"Final WCSS: {model.summary.trainingCost:,.2f}")
print("Canonical cluster mapping (raw -> canonical):")
print(dict(sorted(canonical_mapping.items())))
df_clustered.groupBy("cluster_id").count().orderBy("cluster_id").show()


---
## Step 5: Segment Profiling & Labeling

Analyze cluster characteristics and assign meaningful business labels.

In [ ]:
print("=" * 60)
print("CANONICAL SEGMENT PROFILING")
print("=" * 60)

df_cluster_profiles = (
    df_clustered.groupBy("cluster_id")
    .agg(
        F.count("*").alias("customer_count"),
        F.mean("recency_days").alias("avg_recency"),
        F.mean("frequency").alias("avg_frequency"),
        F.mean("monetary_value").alias("avg_monetary"),
        F.mean("avg_order_value").alias("avg_order_value"),
        F.percentile_approx("recency_days", 0.5, 10000).alias(
            "median_recency"
        ),
        F.percentile_approx("frequency", 0.5, 10000).alias(
            "median_frequency"
        ),
        F.percentile_approx("monetary_value", 0.5, 10000).alias(
            "median_monetary"
        ),
    )
    .orderBy("cluster_id")
)

df_cluster_profiles.show(truncate=False)
recency_percentiles = df_clustered.approxQuantile(
    "recency_days", [0.33, 0.67], 0.0
)
frequency_percentiles = df_clustered.approxQuantile(
    "frequency", [0.33, 0.67], 0.0
)
monetary_percentiles = df_clustered.approxQuantile(
    "monetary_value", [0.33, 0.67], 0.0
)


In [ ]:
def assign_segment_label(
    avg_recency,
    avg_frequency,
    avg_monetary,
    recency_p33,
    recency_p67,
    frequency_p33,
    frequency_p67,
    monetary_p33,
    monetary_p67,
):
    """Assign a business label from a canonical cluster's RFM profile."""
    recency_score = (
        3 if avg_recency < recency_p33
        else 2 if avg_recency < recency_p67
        else 1
    )
    frequency_score = (
        3 if avg_frequency > frequency_p67
        else 2 if avg_frequency > frequency_p33
        else 1
    )
    monetary_score = (
        3 if avg_monetary > monetary_p67
        else 2 if avg_monetary > monetary_p33
        else 1
    )

    if recency_score == 3 and frequency_score == 3 and monetary_score == 3:
        return "Champions"
    if recency_score == 3 and frequency_score >= 2 and monetary_score >= 2:
        return "Loyal Customers"
    if recency_score == 3 and frequency_score <= 2 and monetary_score <= 2:
        return "New Customers"
    if recency_score == 2 and frequency_score >= 2 and monetary_score >= 2:
        return "Potential Loyalists"
    if recency_score == 2:
        return "At Risk"
    if recency_score == 1 and avg_recency > 180:
        return "Lost"
    return "Hibernating"


segment_mapping = []
for row in df_cluster_profiles.orderBy("cluster_id").collect():
    segment_label = assign_segment_label(
        row["avg_recency"],
        row["avg_frequency"],
        row["avg_monetary"],
        recency_percentiles[0],
        recency_percentiles[1],
        frequency_percentiles[0],
        frequency_percentiles[1],
        monetary_percentiles[0],
        monetary_percentiles[1],
    )
    segment_mapping.append((int(row["cluster_id"]), segment_label))
    print(f"Canonical cluster {row['cluster_id']}: {segment_label}")

df_segment_mapping = spark.createDataFrame(
    segment_mapping,
    ["cluster_id", "segment_label"],
)


---
## Step 6: Create Output Table

Join segment labels and save to customer_segments.

In [ ]:
print("=" * 60)
print("CREATING ONE-ROW-PER-CUSTOMER OUTPUT")
print("=" * 60)

df_output = (
    df_clustered.join(df_segment_mapping, "cluster_id", "left")
    .select(
        F.col("customer_id").cast("long"),
        F.col("cluster_id").cast("int"),
        "segment_label",
        "recency_days",
        "frequency",
        "monetary_value",
        "avg_order_value",
        "first_purchase_date",
        "last_purchase_date",
        F.lit(analysis_timestamp).cast("timestamp").alias("segmented_at"),
    )
    .orderBy("customer_id")
)
assert_unique_keys(df_output, ["customer_id"], "Customer segment output")

df_output.groupBy("segment_label").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("monetary_value"), 2).alias("avg_ltv"),
    F.round(F.avg("frequency"), 1).alias("avg_frequency"),
    F.round(F.avg("recency_days"), 1).alias("avg_recency_days"),
).orderBy(F.desc("avg_ltv")).show(truncate=False)


---
## Step 7: Segment Insights & Recommendations

Provide actionable insights for each segment.

In [ ]:
final_evaluator = ClusteringEvaluator(
    predictionCol="cluster_id",
    featuresCol="features",
    metricName="silhouette",
)
final_silhouette = float(final_evaluator.evaluate(df_clustered))
final_wcss = float(model.summary.trainingCost)

with mlflow.start_run(
    run_name=f"kmeans_{analysis_timestamp.strftime('%Y%m%d_%H%M')}"
) as run:
    df_output = (
        df_output.withColumn("generated_at", F.current_timestamp())
        .withColumn("model_run_id", F.lit(run.info.run_id).cast("string"))
        .withColumn("schema_version", F.lit("1.0").cast("string"))
    )
    df_output = validate_ml_output(df_output, "customer_segments")
    save_gold(df_output, "customer_segments")
    mlflow.log_params({
        "algorithm": "kmeans",
        "min_k": MIN_K,
        "max_k": max_candidate_k,
        "k_selection_method": "highest_silhouette_lower_k_tie_break",
        "cluster_id_method": "lexicographically_sorted_scaled_centroids",
        "optimal_k": OPTIMAL_K,
        "random_seed": RANDOM_SEED,
        "source_as_of": analysis_timestamp.isoformat(),
    })
    segment_count = df_output.count()
    mlflow.log_metrics({
        "optimal_k": OPTIMAL_K,
        "silhouette_score": final_silhouette,
        "wcss": final_wcss,
        "customers_segmented": segment_count,
    })
    mlflow.spark.log_model(model, "kmeans_customer_segmentation")
    print(f"MLflow run: {run.info.run_id}")
    print(
        f"Silhouette: {final_silhouette:.4f}, K={OPTIMAL_K}, "
        f"Customers: {segment_count}"
    )


In [ ]:
print("="*60)
print("SEGMENT INSIGHTS & RECOMMENDATIONS")
print("="*60)

recommendations = {
    "Champions": "Reward with VIP benefits, early access, exclusive offers. Encourage referrals.",
    "Loyal Customers": "Upsell higher-value products, loyalty programs, personalized recommendations.",
    "Potential Loyalists": "Nurture with engagement campaigns, increase purchase frequency incentives.",
    "New Customers": "Onboarding programs, welcome offers, build engagement early.",
    "At Risk": "Re-engagement campaigns, special discounts, win-back offers.",
    "Hibernating": "Aggressive win-back campaigns, limited-time offers, surveys for feedback.",
    "Lost": "Final retention attempt with deep discounts or surveys to understand churn."
}

segment_summary = df_output.groupBy("segment_label").count().collect()

print("\n")
for row in segment_summary:
    segment = row["segment_label"]
    count = row["count"]
    recommendation = recommendations.get(segment, "No recommendation available")
    
    print(f"**{segment}** ({count:,} customers)")
    print(f"  - {recommendation}")
    print()

print("="*60)
print("CUSTOMER SEGMENTATION COMPLETE")
print("="*60)

In [ ]:
# Release the ML output lock (see job_locks).
JOB_LOCKS.assert_held()
JOB_LOCKS.release()